In [1]:
import pandas as pd
import pathlib
import os

os.chdir("../..")

# Sorry for this shitty path variables, It will be fixed on server soon (fix for this error: "Permission denied: '/root/.cargo/bin'")
os.environ["PATH"] = ":".join(
    p for p in os.environ["PATH"].split(":")
    if not p.startswith("/root")
)


import pymaid

catmaid_url = 'https://l1em.catmaid.virtualflybrain.org'
http_user = None
http_password = None
project_id = 1

rm = pymaid.CatmaidInstance(catmaid_url, http_user, http_password, project_id)

INFO  : Global CATMAID instance set. Caching is ON. (pymaid)


In [2]:
# -------------------- get ALL neuron ids --------------------
neuron_ids = sorted(map(int, pymaid.get_skids_by_annotation('mw brain and inputs')))

len(neuron_ids)

INFO  : Cached data used. Use `pymaid.clear_cache()` to clear. (pymaid)


3016

## INPUT

### Cell Types
- **sensory** — input нейроны; первичные сенсорные клетки, получающие сигнал из окружающей среды  
- **ascending** — восходящие нейроны из VNC в мозг (обратная связь от тела)  

### Annotations
- **visual** — зрение  
- **olfactory** — обоняние  
- **gustatory-external** — внешний вкус  
- **gustatory-pharyngeal** — глоточный вкус  
- **thermo-warm, thermo-cold** — температурная чувствительность  
- **noci** — ноцицепция (боль)  
- **mechano-Ch** — механорецепция (хордотональные органы)  
- **mechano-II/III** — механорецепция  
- **proprio** — проприоцепция  
- **gut** — кишечная чувствительность  
- **respiratory** — дыхательные сигналы  

---

## OUTPUT (possible)

### Cell Types
- **DN-VNC** — Descending Neurons → Ventral Nerve Cord; основные кандидаты на output, аксоны идут прямо в VNC к мотонейронам  
- **DN-SEZ** — Descending Neurons → Subesophageal Zone; управление ротовым аппаратом и глоткой  
- **pre-DN-SEZ** — Pre-Descending Neurons → SEZ; одно синаптическое звено перед DN-SEZ  
- **pre-DN-VNC** — Pre-Descending Neurons → VNC; одно синаптическое звено перед DN-VNC  

In [3]:
s2 = pd.read_csv('./Datasets/Original/s2.csv')
s3 = pd.read_csv('./Datasets/Original/s3.csv')
s4 = pd.read_csv('./Datasets/Original/s4.csv')
out = pd.read_csv('./Datasets/Original/outputs.csv', index_col=0)



# копируем нужные столбцы
left = s2[['left_id', 'celltype', 'additional_annotations']].copy()
right = s2[['right_id', 'celltype', 'additional_annotations']].copy()

# переименовываем ключ
left = left.rename(columns={'left_id': 'neuron_id'})
right = right.rename(columns={'right_id': 'neuron_id'})

# объединяем
s2_merged = pd.concat([left, right], ignore_index=True)

# убираем строки без нейрона
s2_merged = s2_merged[s2_merged['neuron_id'] != 'no pair']

# приводим тип
s2_merged['neuron_id'] = s2_merged['neuron_id'].astype(int)


# -------------------- load INPUT neurons --------------------
input_neurons = s2_merged[s2_merged['celltype'].isin(['sensory', 'ascending'])]["neuron_id"]
input_neurons = set(input_neurons)


# -------------------- load OUTPUT neurons --------------------
# # we can take it via length of an axon
# output_neurons = set(out[out['axon_output'] > 50].index.astype(int))
output_neurons = s2_merged[s2_merged['celltype'].isin(['DN-VNC', 'DN-SEZ', 'pre-DN-SEZ', 'pre-DN-VNC'])]["neuron_id"]
output_neurons = set(output_neurons)


# -------------------- Build final table --------------------
df = pd.DataFrame({'neuron_id': neuron_ids})
df = df.merge(s2_merged, on='neuron_id', how='left')

# add IO column
IO = []

for nid in df['neuron_id']:
    if nid in input_neurons:
        IO.append('input')
    elif nid in output_neurons:
        IO.append('output')
    else:
        IO.append(None)

df['IO'] = IO

df.to_csv("./Datasets/Generated/IO_Neurons.csv")

In [4]:
print(f'Input Neurons: {len(df[df["IO"] == "input"])}')
print(f'Output Neurons: {len(df[df["IO"] == "output"])}')

df

Input Neurons: 480
Output Neurons: 922


,neuron_id,celltype,additional_annotations,IO
0,29,KC,KC,None
1,37365,sensory,visual,input
2,40045,sensory,olfactory,input
3,40152,sensory,gut,input
4,677717,KC,KC,None
...,...,...,...,...
3011,21590978,NaN,NaN,None
3012,21591033,PN-somato,mechano-Ch 2nd_order PN,None
3013,21591037,pre-DN-VNC,no official annotation,output
3014,21591317,NaN,NaN,None
